In [1]:
# ── 0. Install dependencies if needed ──────────────────────────────────────
# Uncomment the line below the first time you run this notebook
# !pip install geopandas pandas shapely

In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

print(f"pandas  {pd.__version__}")
print(f"geopandas {gpd.__version__}")

pandas  2.3.3
geopandas 1.1.3


## 1  Load & filter the conflict data

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, '.')
from utils import DATA_DIR

# ── Paths ──────────────────────────────────────────────────────────────────
CONFLICT_CSV  = DATA_DIR / 'conflict_data_cod.csv'
SHAPEFILE     = DATA_DIR / 'cod_admin_boundaries.shp/cod_admin2.shp'
OUTPUT_CSV    = DATA_DIR / 'conflict_dat_cleaned.csv'

# ── Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv(CONFLICT_CSV)
print(f"Raw rows : {len(df):,}")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")


Raw rows : 8,901
Year range: 1989 – 2024


In [4]:
# ── Filter 2021-2026 ───────────────────────────────────────────────────────
df = df[df["year"].between(2021, 2026)].copy()
df = df.reset_index(drop=True)
print(f"Rows after year filter (2021–2026): {len(df):,}")
df.head(3)

Rows after year filter (2021–2026): 3,619


,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low,gwnoa,gwnob
0,425460,UGA-2021-1-689-0,2021,False,Clear,1,314,314,Uganda: Government,689,...,2021-12-20 00:00:00.000,0,2,0,0,2,2,2,500.0,NaN
1,375180,DRC-2021-3-934-0,2021,True,Clear,3,89,467,Government of DR Congo (Zaire) - Civilians,89,...,2021-01-08 00:00:00.000,0,0,1,0,1,1,1,490.0,NaN
2,384875,DRC-2021-3-934-5,2021,True,Clear,3,89,467,Government of DR Congo (Zaire) - Civilians,89,...,2021-01-19 00:00:00.000,0,0,1,0,1,1,1,490.0,NaN


## 2  Load the Admin-2 shapefile

> **Note on the shapefile choice**: `cod_admin2.shp` is the DRC Admin-2 layer,
> which represents *territories / zones* (roughly county-level), **not** individual
> towns/cities. If you need true town-level matching you would want a populated-places
> dataset (e.g. OCHA COD-PS or Natural Earth `ne_10m_populated_places`).
> The Admin-2 polygons are nonetheless the standard reference for conflict
> geo-coding in DRC and are what ACLED/UCDP themselves use, so this file is
> appropriate for your analysis. The resulting column `town_admin2` will contain
> the Admin-2 territory/zone name.

In [5]:
# ── Load shapefile ─────────────────────────────────────────────────────────
adm2 = gpd.read_file(SHAPEFILE)
print(f"Shapefile CRS : {adm2.crs}")
print(f"Polygons      : {len(adm2):,}")
print(f"Columns       : {adm2.columns.tolist()}")
adm2.head(3)

Shapefile CRS : EPSG:4326
Polygons      : 164
Columns       : ['adm2_name', 'adm2_name1', 'adm2_name2', 'adm2_name3', 'adm2_pcode', 'adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode', 'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode', 'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1', 'lang2', 'lang3', 'geometry']


,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,adm0_pcode,valid_on,valid_to,area_sqkm,version,lang,lang1,lang2,lang3,geometry
0,Kinshasa,None,None,None,CD1000,Kinshasa,None,None,None,CD10,...,CD,2019-09-11,NaT,10614.813004,v01,fr,None,None,None,"POLYGON ((15.4069 -4.55378, 15.40731 -4.55417,..."
1,Matadi,None,None,None,CD2001,Kongo-Central,None,None,None,CD20,...,CD,2019-09-11,NaT,110.374275,v01,fr,None,None,None,"POLYGON ((13.44006 -5.7931, 13.44313 -5.79194,..."
2,Boma,None,None,None,CD2002,Kongo-Central,None,None,None,CD20,...,CD,2019-09-11,NaT,139.062138,v01,fr,None,None,None,"POLYGON ((13.10054 -5.87856, 13.10022 -5.87767..."


In [6]:
# ── Identify the name column ───────────────────────────────────────────────
# Common Admin-2 name fields in COD shapefiles; we pick the first one found.
CANDIDATE_NAME_COLS = ["admin2Name", "ADM2_EN", "ADM2_FR", "NAME_2",
                       "name", "NAME", "adm2_name", "admin2name"]

name_col = next((c for c in CANDIDATE_NAME_COLS if c in adm2.columns), None)

if name_col is None:
    # Fallback: print all columns so you can pick manually
    raise ValueError(
        f"Could not auto-detect an Admin-2 name column.\n"
        f"Available columns: {adm2.columns.tolist()}\n"
        f"Set `name_col` manually below."
    )

print(f"Using name column: '{name_col}'")
print(adm2[name_col].head(10).tolist())

Using name column: 'adm2_name'
['Kinshasa', 'Matadi', 'Boma', 'Moanda', 'Lukula', 'Tshela', 'Seke-Banza', 'Luozi', 'Songololo', 'Mbanza-Ngungu']


## 3  Spatial join – point-in-polygon

In [7]:
# ── Drop rows with missing coordinates ────────────────────────────────────
n_before = len(df)
df = df.dropna(subset=["latitude", "longitude"]).copy()
print(f"Dropped {n_before - len(df)} rows with missing lat/lon; {len(df):,} remain.")

Dropped 0 rows with missing lat/lon; 3,619 remain.


In [8]:
# ── Build a GeoDataFrame from the conflict points ─────────────────────────
geometry = [Point(lon, lat) for lon, lat in zip(df["longitude"], df["latitude"])]
gdf_pts = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Make sure the Admin-2 layer is in the same CRS
if adm2.crs.to_epsg() != 4326:
    adm2 = adm2.to_crs(epsg=4326)

print("Points CRS :", gdf_pts.crs)
print("Polygons CRS:", adm2.crs)

Points CRS : EPSG:4326
Polygons CRS: EPSG:4326


In [9]:
# ── Spatial join ──────────────────────────────────────────────────────────
# 'predicate="within"' keeps only points inside a polygon.
# Points that fall outside ALL polygons (border edge cases) will have NaN.

joined = gpd.sjoin(
    gdf_pts,
    adm2[[name_col, "geometry"]],
    how="left",
    predicate="within"
)

# Rename the Admin-2 name to a clear column
joined = joined.rename(columns={name_col: "town_admin2"})

# Drop GeoDataFrame helpers we no longer need
joined = joined.drop(columns=["geometry", "index_right"], errors="ignore")

# Back to a plain DataFrame
result = pd.DataFrame(joined)

matched   = result["town_admin2"].notna().sum()
unmatched = result["town_admin2"].isna().sum()
print(f"Matched   : {matched:,} events ({matched/len(result)*100:.1f}%)")
print(f"Unmatched : {unmatched:,} events  (point outside any polygon — likely border coords)")

Matched   : 3,618 events (100.0%)


Unmatched : 1 events  (point outside any polygon — likely border coords)


In [10]:
# ── Quick sense-check ─────────────────────────────────────────────────────
print(result[["year", "latitude", "longitude", "adm_2", "town_admin2"]].head(10))

   year  latitude  longitude               adm_2 town_admin2
0  2021  0.678650  29.624180      Beni territory       Oïcha
1  2021  0.150000  29.283333    Lubero territory     Butembo
2  2021 -2.411250  28.802060    Kabare territory      Kabare
3  2021 -1.275080  29.161310  Rutshuru territory    Rutshuru
4  2021  0.095490  28.980840    Lubero territory      Lubero
5  2021 -4.222570  28.996580      Fizi territory        Fizi
6  2021 -1.237242  29.152844  Rutshuru territory    Rutshuru
7  2021  0.828960  29.432430      Beni territory       Oïcha
8  2021 -0.995240  29.152730  Rutshuru territory    Rutshuru
9  2021  0.718640  29.544890      Beni territory       Oïcha


In [11]:
# Top 10 Admin-2 territories by event count
result["town_admin2"].value_counts().head(10)

town_admin2
Oïcha       666
Djugu       634
Irumu       633
Rutshuru    399
Masisi      233
Mambasa     179
Lubero      118
Mahagi      114
Walikale     80
Fizi         63
Name: count, dtype: int64

## 4  Export

In [12]:
# ── Create output directory if it doesn't exist ───────────────────────────
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

result.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(result):,} rows → {OUTPUT_CSV}")

Saved 3,619 rows → /Users/jackzipper/QSS20/final_project/final_project_data/conflict_dat_cleaned.csv
